In [1]:
!python3 -V
import pDESy
from IPython.display import Markdown
from pDESy.model.base_project import BaseProject
from pDESy.model.base_project import datetime
from pDESy.model.base_product import BaseProduct
from pDESy.model.base_component import BaseComponent
from pDESy.model.base_workflow import BaseWorkflow
from pDESy.model.base_task import BaseTask
from pDESy.model.base_task import BaseTaskDependency
from pDESy.model.base_team import BaseTeam
from pDESy.model.base_worker import BaseWorker
from pDESy.model.base_facility import BaseFacility
from pDESy.model.base_workplace import BaseWorkplace
from pDESy.model.base_priority_rule import TaskPriorityRuleMode,ResourcePriorityRuleMode

Python 3.13.5


In [2]:
pDESy.__version__
pDESy.__file__

'/Users/keisukehirukawa/dev/pDESy_v0.7.3/pDESy/__init__.py'

製品定義

In [3]:
project = BaseProject("sample_workflow")
project = BaseProject(init_datetime = datetime.datetime(2025, 1, 1, 0, 0, 0), unit_timedelta=datetime.timedelta(minutes=60))

# product = project.create_product("product")
for i in range(1):
    command = f"product = project.create_product('product')"
    exec(command)

A = product.create_component("A")
B = product.create_component("B")

ワークフロー定義・同一ワークフロー間依存関係

In [4]:
workflowA = project.create_workflow("workflowA")

task_A1 = workflowA.create_task("A1", need_facility=True, default_work_amount=24.0)
task_A2 = workflowA.create_task("A2", need_facility=True, default_work_amount=24.0)
task_A3 = workflowA.create_task("A3", need_facility=True, default_work_amount=24.0)

A.update_targeted_task_set({task_A1, task_A2, task_A3})

task_A2.add_input_task(task_A1)
task_A3.add_input_task(task_A2)

In [5]:
workflowB = project.create_workflow("workflowB")

task_B1 = workflowB.create_task("B1", need_facility=True, default_work_amount=24.0)
task_B2 = workflowB.create_task("B2", need_facility=True, default_work_amount=72.0)
task_B3 = workflowB.create_task("B3", need_facility=True, default_work_amount=24.0)

B.update_targeted_task_set({task_B1, task_B2, task_B3})

task_B2.add_input_task(task_B1)
task_B3.add_input_task(task_B2)

異なるワークフロー間依存関係

In [6]:
task_B1.add_input_task(task_A2)

設備・人員

In [7]:
# wrokplace model
placeA = project.create_workplace("placeA", max_space_size=10.0)
placeB = project.create_workplace("placeB", max_space_size=10.0)

facilityA = placeA.create_facility("facilityA", cost_per_time=1)
facilityB = placeB.create_facility("facilityB", cost_per_time=1)
facilityB_2 = placeB.create_facility("facilityB_2", cost_per_time=1)
facilityA.workamount_skill_mean_map = {task_A1.name:1.0,task_A2.name:1.0} 
facilityA.workamount_skill_mean_map.update({task_A3.name:1.0})
facilityB.workamount_skill_mean_map = {task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0} 
facilityB_2.workamount_skill_mean_map = facilityB.workamount_skill_mean_map.copy()
# facilityB_2.absence_time_list = [1.0,2.0,3.0,4.0,5.0,6.0]

#team model
team = project.create_team("team1")

wA = team.create_worker("workerA", cost_per_time=10.0, work_constraint_list=[(24,8, "FIXED"), (168,40, "FIXED"), (24,14, "SLIDING"), (168,72, "SLIDING")], rest_constraint_list=[(24,6)])
wB = team.create_worker("workerB", cost_per_time=10.0, work_constraint_list=[(24,8, "FIXED"), (168,40, "FIXED"), (24,14, "SLIDING"), (168,72, "SLIDING")], rest_constraint_list=[(24,6)])

wA.workamount_skill_mean_map = {
    task_A1.name:1.0,task_A2.name:1.0, task_A3.name:1.0,
    task_B1.name:1.0,task_B2.name:1.0,
    } 
wB.workamount_skill_mean_map = {
    task_A1.name:1.0,task_A2.name:1.0,
    task_B1.name:1.0,task_B2.name:1.0, task_B3.name:1.0
    }

wA.facility_skill_map = {facilityA.name:1.0, facilityB.name:1.0, facilityB_2.name:1.0}
wB.facility_skill_map = wA.facility_skill_map.copy()

team.update_targeted_task_set({task_A1,task_A2,task_A3, task_B1,task_B2,task_B3})

placeA.update_targeted_task_set({task_A1,task_A2,task_A3})
placeB.update_targeted_task_set({task_B1,task_B2,task_B3})

In [8]:
print(facilityA.workamount_skill_mean_map)

{'A1': 1.0, 'A2': 1.0, 'A3': 1.0}


In [17]:
print(wA.workamount_skill_mean_map)
print(wA.facility_skill_map)

{'A1': 1.0, 'A2': 1.0, 'A3': 1.0, 'B1': 1.0, 'B2': 1.0}
{'facilityA': 1.0, 'facilityB': 1.0, 'facilityB_2': 1.0}


In [10]:
project.simulate(max_time=600, progress_bar=True)

Completed:  69%|██████▊   | 412/600 [00:00<00:00, 20530.27time/s] 


In [11]:
workflowA.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [12]:
workflowB.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [13]:
from plotly.figure_factory import create_gantt

task_id_list = [
    task_A1.ID, task_A2.ID, task_A3.ID,
    task_B1.ID, task_B2.ID, task_B3.ID
]

all_workflows = [
    workflowA,
    workflowB
]

combined_df = []
for wf in all_workflows:
    combined_df.extend(
        wf.create_data_for_gantt_plotly(
            init_datetime=project.init_datetime,
            unit_timedelta=project.unit_timedelta,
            target_id_order_list=list(task_id_list),
            print_workflow_name=True,
            view_ready=False,           # READY も表示したいなら True
            finish_margin=1.0
        )
    )

colors = {"WORKING": "rgb(146, 237, 5)", "READY": "rgb(107,127,135)"}

fig = create_gantt(
    combined_df,
    title="All Workflows Gantt",
    colors=colors,
    index_col="State",
    showgrid_x=True,
    showgrid_y=True,
    group_tasks=True,
    show_colorbar=True,
)

fig.show()

In [14]:
team.create_gantt_plotly(project.init_datetime, project.unit_timedelta).show()

In [15]:
diagram = "flowchart TB\n" + "\n".join(project.get_mermaid_diagram())
display(Markdown(f"```mermaid\n{diagram}\n```"))

```mermaid
flowchart TB
subgraph 39f275fc-3957-4289-ad6d-ff505310563c[product]
direction LR
28955c05-dfe0-44d2-a945-53ebafe0d55c@{shape: odd, label: 'B'}
810538c0-dc25-4f77-8440-a7f2bae67bca@{shape: odd, label: 'A'}
end
subgraph 146f2d31-d92c-4017-ac4f-794d80f4c007[workflowA]
direction LR
76db59c6-3dd2-4cce-ba13-cbaa1f2648c9@{shape: rect, label: 'A3<br>0.0'}
aa39ba06-92ae-48cc-bc4c-32af45af3455@{shape: rect, label: 'A1<br>0.0'}
11dc79a2-b312-40fb-930b-0c05f58eed29@{shape: rect, label: 'A2<br>0.0'}
11dc79a2-b312-40fb-930b-0c05f58eed29-->76db59c6-3dd2-4cce-ba13-cbaa1f2648c9
aa39ba06-92ae-48cc-bc4c-32af45af3455-->11dc79a2-b312-40fb-930b-0c05f58eed29
end
subgraph 55fec9c6-2e6b-40ee-9787-b01baac09fb0[workflowB]
direction LR
7bd939f7-232a-47da-a98b-7fc9e2e19849@{shape: rect, label: 'B1<br>0.0'}
618d3bff-1881-4375-8e93-0c5f638ceb70@{shape: rect, label: 'B2<br>0.0'}
75a4e849-aa43-4c97-a912-57c58df4b46b@{shape: rect, label: 'B3<br>0.0'}
7bd939f7-232a-47da-a98b-7fc9e2e19849-->618d3bff-1881-4375-8e93-0c5f638ceb70
618d3bff-1881-4375-8e93-0c5f638ceb70-->75a4e849-aa43-4c97-a912-57c58df4b46b
end
subgraph 38d268ee-f222-4eab-94f7-47e7f936b197[team1]
direction LR
68f5336b-0f4d-423d-a2fa-41e6e75250f6@{shape: stadium, label: 'workerB'}
32cdf21a-322b-411f-9cd7-eb7a087c7340@{shape: stadium, label: 'workerA'}
end
subgraph 303df683-693d-4353-9cce-65680f78bcd1[placeA]
direction LR
d2065c4c-1f76-467a-9a85-375bc963e075@{shape: stadium, label: 'facilityA'}
end
subgraph c0cd74fa-38b2-41e6-ac60-190a9643af70[placeB]
direction LR
053448d1-3477-454d-b7d3-1ae70a60b4e8@{shape: stadium, label: 'facilityB_2'}
13ef8f36-d270-4416-a73a-766b5b1357c7@{shape: stadium, label: 'facilityB'}
end
28955c05-dfe0-44d2-a945-53ebafe0d55c-.-75a4e849-aa43-4c97-a912-57c58df4b46b
28955c05-dfe0-44d2-a945-53ebafe0d55c-.-618d3bff-1881-4375-8e93-0c5f638ceb70
28955c05-dfe0-44d2-a945-53ebafe0d55c-.-7bd939f7-232a-47da-a98b-7fc9e2e19849
810538c0-dc25-4f77-8440-a7f2bae67bca-.-76db59c6-3dd2-4cce-ba13-cbaa1f2648c9
810538c0-dc25-4f77-8440-a7f2bae67bca-.-aa39ba06-92ae-48cc-bc4c-32af45af3455
810538c0-dc25-4f77-8440-a7f2bae67bca-.-11dc79a2-b312-40fb-930b-0c05f58eed29
76db59c6-3dd2-4cce-ba13-cbaa1f2648c9-.-38d268ee-f222-4eab-94f7-47e7f936b197
303df683-693d-4353-9cce-65680f78bcd1-.-76db59c6-3dd2-4cce-ba13-cbaa1f2648c9
aa39ba06-92ae-48cc-bc4c-32af45af3455-.-38d268ee-f222-4eab-94f7-47e7f936b197
303df683-693d-4353-9cce-65680f78bcd1-.-aa39ba06-92ae-48cc-bc4c-32af45af3455
11dc79a2-b312-40fb-930b-0c05f58eed29-.-38d268ee-f222-4eab-94f7-47e7f936b197
303df683-693d-4353-9cce-65680f78bcd1-.-11dc79a2-b312-40fb-930b-0c05f58eed29
7bd939f7-232a-47da-a98b-7fc9e2e19849-.-38d268ee-f222-4eab-94f7-47e7f936b197
c0cd74fa-38b2-41e6-ac60-190a9643af70-.-7bd939f7-232a-47da-a98b-7fc9e2e19849
618d3bff-1881-4375-8e93-0c5f638ceb70-.-38d268ee-f222-4eab-94f7-47e7f936b197
c0cd74fa-38b2-41e6-ac60-190a9643af70-.-618d3bff-1881-4375-8e93-0c5f638ceb70
75a4e849-aa43-4c97-a912-57c58df4b46b-.-38d268ee-f222-4eab-94f7-47e7f936b197
c0cd74fa-38b2-41e6-ac60-190a9643af70-.-75a4e849-aa43-4c97-a912-57c58df4b46b
11dc79a2-b312-40fb-930b-0c05f58eed29-->7bd939f7-232a-47da-a98b-7fc9e2e19849
```